# 06 — Suspicious-pattern and graph screening

Screens observed source patterns; it does not label AML typologies or inject records.

In [1]:
# Setup
from pathlib import Path
import logging, random
import numpy as np
import pandas as pd
random.seed(42); np.random.seed(42)
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
roots = (Path('/kaggle/input'), Path('data/original'))
edge_paths = [p for root in roots if root.exists() for p in root.rglob('graph_edges.csv')]
feature_paths = [p for root in roots if root.exists() for p in root.rglob('ml_features.csv')]
if not edge_paths or not feature_paths: raise FileNotFoundError('Upload the source bundle to Kaggle Input or data/original.')
edges = pd.read_csv(edge_paths[0])
features = pd.read_csv(feature_paths[0])


In [2]:
pair_counts = edges.groupby(['Sender_account', 'Receiver_account']).size().rename('edge_count')
sender_degree = edges.groupby('Sender_account').size().rename('outgoing_transactions')
receiver_degree = edges.groupby('Receiver_account').size().rename('incoming_transactions')
print('Unique directed pairs:', len(pair_counts))
print('Repeated directed pairs:', (pair_counts > 1).sum())
display(sender_degree.sort_values(ascending=False).head(20).to_frame())
display(receiver_degree.sort_values(ascending=False).head(20).to_frame())
threshold = features.loc[features['above_1M_NPR'].eq(1), ['amount_local_npr', 'is_suspicious_tx']]
display(threshold.groupby('is_suspicious_tx')['amount_local_npr'].describe())
display(features.groupby('is_suspicious_tx')[['cross_border_flag', 'currency_mismatch', 'tx_count_10', 'tx_count_30']].mean())


Unique directed pairs: 50586
Repeated directed pairs: 8440


,outgoing_transactions
Sender_account,
8976725341,265
382301928,264
15297964,263
1452170043,262
7658664999,262
6984051591,262
3117792558,261
7281632640,261
9067453186,247


,incoming_transactions
Receiver_account,
9683990807,241
6086421020,241
283280424,240
464342049,240
7099141711,240
4063749322,240
1491736788,240
2342866371,240
5251195293,240


,count,mean,std,min,25%,50%,75%,max
is_suspicious_tx,,,,,,,,
0,59313.0,2.713045e+06,5.893531e+06,1000036.88,1.393990e+06,1896777.810,2.780129e+06,2.053320e+08
1,274.0,6.657588e+06,3.351343e+07,1013973.78,1.831317e+06,1932901.265,7.044716e+06,5.527964e+08


,cross_border_flag,currency_mismatch,tx_count_10,tx_count_30
is_suspicious_tx,,,,
0,0.101025,0.116883,5.075716,9.367759
1,0.163690,0.142857,2.345238,2.375000


## Conclusion
High-degree or repeated edges are screening signals, not proof of mule networks. Formal AML injection and graph modelling are explicitly deferred to later phases.